# Création du modèle IA + entrainement + visualisation des performance

## 1) import

In [ ]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces
from stable_baselines3 import PPO
import matplotlib.pyplot as plt
from stable_baselines3.common.env_util import make_vec_env
import matplotlib.pyplot as plt
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
import os
import torch

## 2) 🧠 ForexEnv — Environnement d’apprentissage pour OrionTrader

Ce fichier décrit la logique du code Python `ForexEnv`, qui simule un petit marché de change (Forex) simplifié.  
Cet environnement est compatible avec **Gymnasium / OpenAI Gym**, ce qui permet d’y entraîner des agents de **reinforcement learning** (comme PPO, DQN, etc.).

### 🧩 Description des composants

#### Initialisation (`__init__`)

- `balance`: capital initial du trader (1000 unités)
- `price`: prix actuel du marché (1.1 EUR/USD)
- `action_space`: trois actions possibles
    - 0 → Acheter
    - 1 → Vendre
    - 2 → Attendre
- `observation_space`: observation unique = prix actuel, bornée entre 0 et 10

#### Réinitialisation (`reset`)

- Remet à zéro l’environnement (capital et prix)
- Retourne l’état initial (ici, juste le prix)
- Utilisé automatiquement par l’agent au début de chaque épisode

#### Évolution d’un pas (`step`)

- `step()` est appelée à chaque action de l’agent.
- Le prix évolue de manière aléatoire (marché simulé avec une petite volatilité de ±1%).

#### Gestion des actions

|Action|Description|Formule de la récompense|
|---|---|---|
|0|Buy : on achète du dollar|(self.price - old_price) * 1000|
|1|Sell : on vend du dollar|(old_price - self.price) * 1000|
|2|Hold : on attend|-0.1 (pénalité d’inaction)|

#### ⚙️ Objectif pour l’agent IA

L’agent (par ex. un modèle PPO) doit apprendre à :

- acheter quand le prix va monter,
- vendre quand le prix va baisser,
- et éviter les pertes (ou les longues périodes d’inaction).

Chaque épisode correspond à une session de trading où il essaie de maximiser sa balance finale.

#### 🚀 Prochaines améliorations possibles
- Ajouter un spread (écart entre prix achat/vente)
- Intégrer un historique de prix (observations multi-dimensionnelles)
- Simuler un effet de levier
- Ajouter une mémoire d’état pour permettre des stratégies plus complexes
- Connecter aux vraies données MT5

In [ ]:
# class ForexEnv(gym.Env):
#     """
#     ForexEnv minimal amélioré :
#     - observation : 3 dernières valeurs de prix
#     - reward : variation relative * position_size
#     - hold a un coût pour encourager l'action
#     """
#     def __init__(self):
#         super(ForexEnv, self).__init__()
#         self.initial_balance = 1000.0
#         self.balance = self.initial_balance
#         self.prices = [1.1, 1.1, 1.1]  # sliding window length 3
#         self.position_size = 1000.0
#         self.action_space = spaces.Discrete(3)  # 0: Buy, 1: Sell, 2: Hold
#         # observation : last 3 prices
#         self.observation_space = spaces.Box(low=0.0, high=10.0, shape=(3,), dtype=np.float32)

#     def reset(self, seed=None, options=None):
#         self.balance = self.initial_balance
#         self.prices = [1.1, 1.1, 1.1]
#         return np.array(self.prices, dtype=np.float32), {}

#     def step(self, action):
#         old_price = self.prices[-1]
#         # simulate market move (toujours random normal pour l'exemple)
#         new_price = float(old_price + np.random.normal(0, 0.01))
#         self.prices.append(new_price)
#         if len(self.prices) > 3:
#             self.prices.pop(0)

#         # reward : relative price change * position size
#         price_change_rel = (new_price - old_price) / old_price  # relative change
#         if action == 0:  # Buy
#             reward = price_change_rel * self.position_size
#         elif action == 1:  # Sell
#             reward = -price_change_rel * self.position_size
#         else:  # Hold
#             reward = -1.0  # coût d'opportunité / frais de maintien

#         # Safety: clip reward to avoid very large spikes
#         reward = float(np.clip(reward, -1e3, 1e3))

#         self.balance += reward
#         done = bool(self.balance <= 0.0 or self.balance >= 2 * self.initial_balance)
#         obs = np.array(self.prices, dtype=np.float32)
#         info = {}
#         return obs, reward, done, False, info

# env = ForexEnv()

In [ ]:
from stable_baselines3.common.callbacks import BaseCallback
import os, csv

class RewardPerStepLogger(BaseCallback):
    """Logger du reward moyen par step depuis ep_info_buffer (TensorBoard + CSV + console)."""
    def __init__(self, verbose=1, csv_path=None):
        super().__init__(verbose)
        self.csv_path = csv_path
        if self.csv_path is not None:
            os.makedirs(os.path.dirname(self.csv_path), exist_ok=True)
            if not os.path.exists(self.csv_path):
                with open(self.csv_path, "w", newline="") as f:
                    writer = csv.writer(f)
                    writer.writerow(["timesteps", "reward_per_step"])

    def _on_step(self) -> bool:
        return True

    def _on_rollout_end(self) -> None:
        # utiliser ep_info_buffer du modèle
        try:
            ep_infos = list(self.model.ep_info_buffer)
        except Exception:
            ep_infos = []

        if len(ep_infos) == 0:
            if self.verbose > 0:
                print("[RewardPerStepLogger] ⚠️ Aucun épisode terminé ce rollout")
            return

        ep_rews = [e["r"] for e in ep_infos]
        ep_lens = [e["l"] for e in ep_infos]
        total_steps = sum(ep_lens)
        reward_per_step = sum(ep_rews) / total_steps if total_steps > 0 else 0.0

        # TensorBoard
        self.logger.record("diagnostic/reward_per_step", float(reward_per_step))
        # CSV
        if self.csv_path is not None:
            with open(self.csv_path, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow([self.num_timesteps, float(reward_per_step)])
        # Console
        if self.verbose > 0:
            print(f"[RewardPerStepLogger] timesteps={self.num_timesteps} reward/step={reward_per_step:.8f}")

# -----------------------
# 3) Helpers utilitaires
# -----------------------
def set_optimizer_lr(model, new_lr: float):
    """Met à jour le learning rate de l'optimizer SB3."""
    for g in model.optimizer.param_groups:
        g["lr"] = new_lr

def set_attr_and_log(model, **kwargs):
    """Met à jour certains attributs du modèle et logge."""
    for k, v in kwargs.items():
        if hasattr(model, k):
            setattr(model, k, v)
    print("[MODEL] Updated attrs:", kwargs)


## 🤖 4. Entraînement du modèle PPO

In [ ]:
# -----------------------
# 4) Configuration & création envs
# -----------------------
SEED = 42
run_name = "orion_finetune"
logdir = f"./logs/{run_name}"
os.makedirs(logdir, exist_ok=True)
os.makedirs(f"./models/{run_name}", exist_ok=True)

# Vectorized env (8 workers recommandé)
n_envs = 8
vec_env = make_vec_env(ForexEnv, n_envs=n_envs, seed=SEED)
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

# Eval env (partage stats de normalisation)
eval_env = make_vec_env(ForexEnv, n_envs=1, seed=SEED+1)
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=True, clip_obs=10.0)
eval_env.obs_rms = vec_env.obs_rms
eval_env.ret_rms = vec_env.ret_rms

# Callbacks
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=f"./models/{run_name}/best/",
    log_path=f"{logdir}/eval/",
    eval_freq=10_000,
    n_eval_episodes=5,
    deterministic=True,
)

checkpoint_callback = CheckpointCallback(
    save_freq=50_000,
    save_path=f"./models/{run_name}/checkpoints/",
    name_prefix="ppo_orion"
)

reward_csv = f"./logs/{run_name}/reward_per_step.csv"
reward_logger = RewardPerStepLogger(verbose=1, csv_path=reward_csv)

In [ ]:
# -----------------------
# 5) Charger ou créer le modèle initial
# -----------------------
START_FROM_EXISTING = False
EXISTING_MODEL_PATH = "./models/orion/final.zip"  # si tu veux partir d'un modèle existant

if START_FROM_EXISTING and os.path.exists(EXISTING_MODEL_PATH):
    print("Loading existing model:", EXISTING_MODEL_PATH)
    model = PPO.load(EXISTING_MODEL_PATH, env=vec_env)
    # ensure optimizer has correct lr (set to a safe starting lr)
    set_optimizer_lr(model, 3e-5)
else:
    print("Creating new PPO model.")
    model = PPO(
        "MlpPolicy",
        vec_env,
        verbose=1,
        tensorboard_log=logdir,
        learning_rate=3e-5,
        n_steps=2048,
        batch_size=256,
        n_epochs=3,
        ent_coef=0.005,
        clip_range=0.15,
        vf_coef=0.7,
        max_grad_norm=0.5,
        target_kl=0.02,
    )

In [ ]:
# # reproducibility
# SEED = 42
# np.random.seed(SEED)
# torch.manual_seed(SEED)

# run_name = "orion"
# logdir = f"./logs/{run_name}"
# os.makedirs(logdir, exist_ok=True)
# os.makedirs(f"./models/{run_name}", exist_ok=True)

# # 1) Vectorized env (4 workers recommended)
# vec_env = make_vec_env(ForexEnv, n_envs=8, seed=SEED)
# vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.)

# # 2) Eval env (share normalization stats afterwards)
# eval_env = make_vec_env(ForexEnv, n_envs=8, seed=SEED+1)
# eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=True, clip_obs=10.)
# eval_env.obs_rms = vec_env.obs_rms  # sync obs normalization

# # 3) Callbacks
# eval_callback = EvalCallback(
#     eval_env,
#     best_model_save_path=f"./models/{run_name}/best/",
#     log_path=f"{logdir}/eval/",
#     eval_freq=10_000,
#     n_eval_episodes=5,
#     deterministic=True,
# )

# checkpoint_callback = CheckpointCallback(
#     save_freq=50_000,
#     save_path=f"./models/{run_name}/checkpoints/",
#     name_prefix="ppo_orion"
# )

# # 4) PPO model with stable hyperparams
# model = PPO(
#     "MlpPolicy",
#     vec_env,
#     verbose=1,
#     tensorboard_log=logdir,
#     learning_rate=3e-5,   # modest LR to avoid big policy jumps
#     n_steps=2048,         # more steps -> smoother gradient estimates
#     batch_size=256,       # must divide n_steps * n_envs (4096*4=16384)
#     n_epochs=3,           # fewer epochs = less overfitting on same rollouts
#     ent_coef=0.003,        # encourage exploration (avoid premature determinism)
#     clip_range=0.15,       # conservative policy updates
#     vf_coef=0.7,          # weight for value loss (default ok)
#     max_grad_norm=0.5,
#     target_kl=0.02,       # optional: stop epoch early if KL too large
# )

# reward_per_step_logger = RewardPerStepLogger(
#     verbose=1, 
#     csv_path=f"./logs/{run_name}/reward_per_step.csv"
# )

# model.learn(
#     total_timesteps=400_000,
#     callback=[eval_callback, checkpoint_callback, reward_per_step_logger]
# )

# # 6) Save final artifacts
# model.save(f"./models/{run_name}/final")
# vec_env.save(f"./models/{run_name}/vecnormalize.pkl")

## 📊 5. Visualiser la progression (TensorBoard)

```bash
tensorboard --logdir ./logs/orion_finetune/
```

-> ouvrir le navigateur à http://localhost:6006

pour y voir :

- Courbes des récompenses
- Longueur moyenne des épisodes
- Entropie (exploration)
- Pertes du modèle

## 🧪 6. Tester le modèle entraîné

In [ ]:
# import numpy as np
# from stable_baselines3 import PPO
# from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# # ---------------------------
# # 1️⃣ Charger le modèle et l'environnement
# # ---------------------------

# # Charger le modèle entraîné
# model = PPO.load("./models/orion/final")

# # Recréer l'environnement pour test
# env = DummyVecEnv([lambda: ForexEnv()])

# # Charger la normalisation utilisée à l'entraînement
# env = VecNormalize.load("./models/orion/vecnormalize.pkl", env)
# env.training = False        # mode évaluation
# env.norm_reward = True      # normaliser les rewards comme à l'entraînement

# # ---------------------------
# # 2️⃣ Variables de suivi
# # ---------------------------

# n_steps_test = 1300000  # nombre d'étapes à tester
# total_reward = 0.0
# obs = env.reset()

# # ---------------------------
# # 3️⃣ Boucle de test
# # ---------------------------

# for step in range(n_steps_test):
#     # prédire l'action
#     action, _ = model.predict(obs, deterministic=True)

#     # appliquer l'action
#     obs, reward, done, info = env.step(action)

#     # reward est un array car VecEnv
#     reward_val = reward[0]
#     total_reward += reward_val

#     # accéder au prix et au balance de l'environnement
#     price = obs[0][0]  # ajuster selon ton obs_space
#     balance = env.get_attr('balance')[0]

#     print(f"Step={step:02d} | Action={action[0]} | Price={price:.4f} | Reward={reward_val:.2f} | Balance={balance:.2f}")

#     if done[0]:
#         print("Fin de l’épisode.")
#         break

# # ---------------------------
# # 4️⃣ Calcul reward par step
# # ---------------------------

# reward_per_step = total_reward / (step + 1)
# print(f"\n🎯 Récompense totale : {total_reward:.2f}")
# print(f"⚡ Reward moyen par step : {reward_per_step:.4f}")


In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# import csv
# from stable_baselines3 import PPO
# from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# def load_model_and_env(model_path, vecnorm_path):
#     model = PPO.load(model_path)
#     env = DummyVecEnv([lambda: ForexEnv()])
#     env = VecNormalize.load(vecnorm_path, env)
#     env.training = False
#     env.norm_reward = True
#     return model, env

# def evaluate(model, env, n_steps=None, n_episodes=None, deterministic=True, csv_out=None):
#     """
#     Evaluate either for n_steps total (stop when steps reached) or n_episodes complete episodes.
#     Returns episode_rewards, episode_lengths, cumulative_balance_series (per step)
#     """
#     assert (n_steps is not None) ^ (n_episodes is not None), "Donner n_steps XOR n_episodes"

#     obs = env.reset()
#     episode_rewards = []
#     episode_lengths = []
#     balances = []   # track balance over time (for the first env)
#     curr_ep_reward = 0.0
#     curr_ep_len = 0
#     total_steps = 0
#     cum_balance = env.get_attr("balance")[0]

#     balances.append(cum_balance)

#     while True:
#         action, _ = model.predict(obs, deterministic=deterministic)
#         obs, reward, done, info = env.step(action)
#         r = float(reward[0])
#         curr_ep_reward += r
#         curr_ep_len += 1
#         total_steps += 1
#         cum_balance = env.get_attr("balance")[0]
#         balances.append(cum_balance)

#         if done[0]:
#             episode_rewards.append(curr_ep_reward)
#             episode_lengths.append(curr_ep_len)
#             curr_ep_reward = 0.0
#             curr_ep_len = 0
#             obs = env.reset()
#             # stop conditions
#             if n_episodes is not None and len(episode_rewards) >= n_episodes:
#                 break
#         if n_steps is not None and total_steps >= n_steps:
#             # if in middle of episode, finish current episode optionally or just stop
#             break

#     # save CSV if requested
#     if csv_out is not None:
#         with open(csv_out, "w", newline="") as f:
#             writer = csv.writer(f)
#             writer.writerow(["episode", "ep_reward", "ep_length"])
#             for i, (r, l) in enumerate(zip(episode_rewards, episode_lengths)):
#                 writer.writerow([i, r, l])
#     return episode_rewards, episode_lengths, balances

# def bootstrap_mean_ci(data, n_boot=5000, alpha=0.05):
#     arr = np.array(data)
#     boot_means = []
#     n = len(arr)
#     for _ in range(n_boot):
#         sample = np.random.choice(arr, size=n, replace=True)
#         boot_means.append(np.mean(sample))
#     low = np.percentile(boot_means, 100*alpha/2)
#     high = np.percentile(boot_means, 100*(1-alpha/2))
#     return np.mean(arr), np.std(arr, ddof=1), (low, high)

# # --------------------------
# # Parameters: adapte ici
# # --------------------------
# MODEL_PATH = "./models/orion/final"
# VECNORM_PATH = "./models/orion/vecnormalize.pkl"
# CSV_OUT_EPISODES = "./logs/orion/eval_episodes.csv"
# N_EPISODES = 50   # par ex. 50 épisodes (ou mettre None et utiliser N_STEPS)
# # N_STEPS = 1300000

# # --------------------------
# # Run
# # --------------------------
# model, env = load_model_and_env(MODEL_PATH, VECNORM_PATH)
# ep_rewards, ep_lens, balances = evaluate(model, env, n_episodes=N_EPISODES, csv_out=CSV_OUT_EPISODES)

# # Stats per episode
# mean_ep_reward = np.mean(ep_rewards)
# std_ep_reward = np.std(ep_rewards, ddof=1)
# mean_ep_len = np.mean(ep_lens)

# # reward per step per episode distribution
# reward_per_step_per_ep = np.array(ep_rewards) / np.array(ep_lens)

# mean_rps, std_rps = np.mean(reward_per_step_per_ep), np.std(reward_per_step_per_ep, ddof=1)
# mean_rps_allsteps = sum(ep_rewards) / sum(ep_lens)

# # bootstrap CI
# mean_rps_mean, std_rps_mean, ci = bootstrap_mean_ci(reward_per_step_per_ep)

# print("=== Résultats d'évaluation ===")
# print(f"episodes: {len(ep_rewards)}")
# print(f"mean ep reward = {mean_ep_reward:.4f} ± {std_ep_reward:.4f}")
# print(f"mean ep length = {mean_ep_len:.1f}")
# print(f"mean reward/step (episode avg) = {mean_rps:.6e} (std {std_rps:.6e})")
# print(f"global reward/step (all steps) = {mean_rps_allsteps:.6e}")
# print(f"Bootstrap 95% CI for mean(reward/step per ep): [{ci[0]:.6e}, {ci[1]:.6e}]")

# # Sharpe-like (episode returns)
# sharpe = (np.mean(ep_rewards) / (np.std(ep_rewards, ddof=1) + 1e-12))
# print(f"Sharpe-like (mean_ep_reward / std_ep_reward) = {sharpe:.4f}")

# # Plot balance
# plt.figure(figsize=(10,4))
# plt.plot(balances)
# plt.title("Balance cumulative au fil des steps")
# plt.xlabel("step")
# plt.ylabel("balance")
# plt.grid(True)
# plt.tight_layout()
# plt.show()


In [ ]:
# # -----------------------
# # 6) Phase de fine-tuning progressive
# # -----------------------
# # Phase A: Warm-up exploration (encourage l'exploration)
# print("\n=== Phase A: Warm-up exploration ===")
# # augmenter ent_coef temporairement et lr raisonnable
# set_attr_and_log(model, ent_coef=0.01, vf_coef=0.7, clip_range=0.15)
# set_optimizer_lr(model, 5e-5)
# model.learn(total_timesteps=300_000, callback=[eval_callback, checkpoint_callback, reward_logger])

# # sauvegarder checkpoint intermédiaire + vecnorm
# model.save(f"./models/{run_name}/after_phase_A")
# vec_env.save(f"./models/{run_name}/vecnormalize.pkl")

# # Phase B: Stabilisation
# print("\n=== Phase B: Stabilisation ===")
# set_attr_and_log(model, ent_coef=0.005, vf_coef=0.75, clip_range=0.18)
# set_optimizer_lr(model, 3e-5)
# model.learn(total_timesteps=500_000, callback=[eval_callback, checkpoint_callback, reward_logger])

# model.save(f"./models/{run_name}/after_phase_B")
# vec_env.save(f"./models/{run_name}/vecnormalize.pkl")

# # Phase C: Exploitation (moins d'exploration, petit lr)
# print("\n=== Phase C: Exploitation ===")
# set_attr_and_log(model, ent_coef=0.001, vf_coef=0.8, clip_range=0.12)
# set_optimizer_lr(model, 1e-5)
# # Note: on ne change pas n_steps ici pour rester compatible
# model.learn(total_timesteps=500_000, callback=[eval_callback, checkpoint_callback, reward_logger])

# # -----------------------
# # 7) Sauvegarde finale
# # -----------------------
# model.save(f"./models/{run_name}/final")
# vec_env.save(f"./models/{run_name}/vecnormalize.pkl")
# print("Fine-tuning terminé. Modèle et VecNormalize sauvegardés.")

In [ ]:
# # -----------------------
# # 7) Sauvegarde finale
# # -----------------------
# model.save(f"./models/{run_name}/final")
# vec_env.save(f"./models/{run_name}/vecnormalize.pkl")
# print("Fine-tuning terminé. Modèle et VecNormalize sauvegardés.")

## 🧭 9. Prochaines étapes possibles

 - 🔧 Ajouter des features de marché réelles (historique MT5)
 - ⚙️ Intégrer OrionTrader à ton API FastAPI → pour qu’il prenne des décisions sur des flux temps réel
 - 📊 Enregistrer les trades dans une base SQLite / CSV
 - 🧠 Optimiser l’entraînement (changer total_timesteps, learning rate…)

In [ ]:
# -----------------------
# 1) Env amélioré
# -----------------------
class ForexEnv(gym.Env):
    """
    ForexEnv minimal amélioré :
    - observation : 3 dernières valeurs de prix
    - reward : variation relative * position_size
    - hold a un coût pour encourager l'action
    """
    def __init__(self):
        super(ForexEnv, self).__init__()
        self.initial_balance = 1000.0
        self.balance = self.initial_balance
        self.prices = [1.1, 1.1, 1.1]  # sliding window length 3
        self.position_size = 1000.0
        self.action_space = spaces.Discrete(3)  # 0: Buy, 1: Sell, 2: Hold
        # observation : last 3 prices
        self.observation_space = spaces.Box(low=0.0, high=10.0, shape=(3,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        self.balance = self.initial_balance
        self.prices = [1.1, 1.1, 1.1]
        return np.array(self.prices, dtype=np.float32), {}

    def step(self, action):
        old_price = self.prices[-1]
        # simulate market move (toujours random normal pour l'exemple)
        new_price = float(old_price + np.random.normal(0, 0.01))
        self.prices.append(new_price)
        if len(self.prices) > 3:
            self.prices.pop(0)

        # reward : relative price change * position size
        price_change_rel = (new_price - old_price) / old_price  # relative change
        if action == 0:  # Buy
            reward = price_change_rel * self.position_size
        elif action == 1:  # Sell
            reward = -price_change_rel * self.position_size
        else:  # Hold
            reward = -1.0  # coût d'opportunité / frais de maintien

        # Safety: clip reward to avoid very large spikes
        reward = float(np.clip(reward, -1e3, 1e3))

        self.balance += reward
        done = bool(self.balance <= 0.0 or self.balance >= 2 * self.initial_balance)
        obs = np.array(self.prices, dtype=np.float32)
        info = {}
        return obs, reward, done, False, info

# -----------------------
# 2) RewardPerStepLogger (CSV + TB)
# -----------------------
class RewardPerStepLogger(BaseCallback):
    """Logger du reward moyen par step depuis ep_info_buffer (TensorBoard + CSV + console)."""
    def __init__(self, verbose=1, csv_path=None):
        super().__init__(verbose)
        self.csv_path = csv_path
        if self.csv_path is not None:
            os.makedirs(os.path.dirname(self.csv_path), exist_ok=True)
            if not os.path.exists(self.csv_path):
                with open(self.csv_path, "w", newline="") as f:
                    writer = csv.writer(f)
                    writer.writerow(["timesteps", "reward_per_step"])

    def _on_step(self) -> bool:
        return True

    def _on_rollout_end(self) -> None:
        # utiliser ep_info_buffer du modèle
        try:
            ep_infos = list(self.model.ep_info_buffer)
        except Exception:
            ep_infos = []

        if len(ep_infos) == 0:
            if self.verbose > 0:
                print("[RewardPerStepLogger] ⚠️ Aucun épisode terminé ce rollout")
            return

        ep_rews = [e["r"] for e in ep_infos]
        ep_lens = [e["l"] for e in ep_infos]
        total_steps = sum(ep_lens)
        reward_per_step = sum(ep_rews) / total_steps if total_steps > 0 else 0.0

        # TensorBoard
        self.logger.record("diagnostic/reward_per_step", float(reward_per_step))
        # CSV
        if self.csv_path is not None:
            with open(self.csv_path, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow([self.num_timesteps, float(reward_per_step)])
        # Console
        if self.verbose > 0:
            print(f"[RewardPerStepLogger] timesteps={self.num_timesteps} reward/step={reward_per_step:.8f}")

# -----------------------
# 3) Helpers utilitaires
# -----------------------
def set_optimizer_lr(model, new_lr: float):
    """
    Met à jour le learning rate de PPO.
    Fonctionne avant ou après la création de l'optimizer.
    """
    # 1️⃣ Met à jour la fonction de LR (utilisée par PPO si optimizer pas encore créé)
    if hasattr(model, "lr_schedule"):
        model.lr_schedule = lambda _: new_lr

    # 2️⃣ Si l'optimizer existe déjà, on change ses param_groups
    if hasattr(model, "optimizer") and hasattr(model.optimizer, "param_groups"):
        for g in model.optimizer.param_groups:
            g["lr"] = new_lr

    print(f"[MODEL] Learning rate mis à jour à {new_lr:.2e}")

def set_attr_and_log(model, **kwargs):
    """Met à jour les hyperparamètres PPO (avec compatibilité SB3)."""
    for k, v in kwargs.items():
        # SB3 attend des schedules pour clip_range, learning_rate, etc.
        if k == "clip_range":
            from stable_baselines3.common.utils import get_schedule_fn
            v = get_schedule_fn(v)
        setattr(model, k, v)
        print(f"[MODEL] {k} mis à jour à {v}")

# -----------------------
# 4) Configuration & création envs
# -----------------------
SEED = 42
run_name = "orion_finetune"
logdir = f"./logs/{run_name}"
os.makedirs(logdir, exist_ok=True)
os.makedirs(f"./models/{run_name}", exist_ok=True)

# Vectorized env (8 workers recommandé)
n_envs = 8
vec_env = make_vec_env(ForexEnv, n_envs=n_envs, seed=SEED)
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

# Eval env (partage stats de normalisation)
eval_env = make_vec_env(ForexEnv, n_envs=1, seed=SEED+1)
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=True, clip_obs=10.0)
eval_env.obs_rms = vec_env.obs_rms
eval_env.ret_rms = vec_env.ret_rms

# Callbacks
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=f"./models/{run_name}/best/",
    log_path=f"{logdir}/eval/",
    eval_freq=10_000,
    n_eval_episodes=5,
    deterministic=True,
)

checkpoint_callback = CheckpointCallback(
    save_freq=50_000,
    save_path=f"./models/{run_name}/checkpoints/",
    name_prefix="ppo_orion"
)

reward_csv = f"./logs/{run_name}/reward_per_step.csv"
reward_logger = RewardPerStepLogger(verbose=1, csv_path=reward_csv)

# -----------------------
# 5) Charger ou créer le modèle initial
# -----------------------
START_FROM_EXISTING = False
EXISTING_MODEL_PATH = "./models/orion/final.zip"  # si tu veux partir d'un modèle existant

if START_FROM_EXISTING and os.path.exists(EXISTING_MODEL_PATH):
    print("Loading existing model:", EXISTING_MODEL_PATH)
    model = PPO.load(EXISTING_MODEL_PATH, env=vec_env)
    # ensure optimizer has correct lr (set to a safe starting lr)
    set_optimizer_lr(model, 3e-5)
else:
    print("Creating new PPO model.")
    model = PPO(
        "MlpPolicy",
        vec_env,
        verbose=1,
        tensorboard_log=logdir,
        learning_rate=3e-5,
        n_steps=2048,
        batch_size=256,
        n_epochs=3,
        ent_coef=0.005,
        clip_range=0.15,
        vf_coef=0.7,
        max_grad_norm=0.5,
        target_kl=0.02,
    )

# -----------------------
# 6) Phase de fine-tuning progressive
# -----------------------

# ✅ Sauvegarde du modèle de base avant toute modification
base_save_path = "./models/orion/base/"
os.makedirs(base_save_path, exist_ok=True)
model.save(f"{base_save_path}/ppo_orion")
print(f"[INFO] Modèle de base sauvegardé dans {base_save_path}")

# Phase A: Warm-up exploration (encourage l'exploration)
print("\n=== Phase A: Warm-up exploration ===")
# augmenter ent_coef temporairement et lr raisonnable
set_attr_and_log(model, ent_coef=0.01, vf_coef=0.7, clip_range=0.15)
set_optimizer_lr(model, 5e-5)
model.learn(total_timesteps=300_000, callback=[eval_callback, checkpoint_callback, reward_logger])

# sauvegarder checkpoint intermédiaire + vecnorm
model.save(f"./models/{run_name}/after_phase_A")
vec_env.save(f"./models/{run_name}/vecnormalize.pkl")

# Phase B: Stabilisation
print("\n=== Phase B: Stabilisation ===")
set_attr_and_log(model, ent_coef=0.005, vf_coef=0.75, clip_range=0.18)
set_optimizer_lr(model, 3e-5)
model.learn(total_timesteps=500_000, callback=[eval_callback, checkpoint_callback, reward_logger])

model.save(f"./models/{run_name}/after_phase_B")
vec_env.save(f"./models/{run_name}/vecnormalize.pkl")

# Phase C: Exploitation (moins d'exploration, petit lr)
print("\n=== Phase C: Exploitation ===")
set_attr_and_log(model, ent_coef=0.001, vf_coef=0.8, clip_range=0.12)
set_optimizer_lr(model, 1e-5)
# Note: on ne change pas n_steps ici pour rester compatible
model.learn(total_timesteps=500_000, callback=[eval_callback, checkpoint_callback, reward_logger])

# -----------------------
# 7) Sauvegarde finale
# -----------------------
model.save(f"./models/{run_name}/final")
vec_env.save(f"./models/{run_name}/vecnormalize.pkl")
print("Fine-tuning terminé. Modèle et VecNormalize sauvegardés.")


In [ ]:
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

def evaluate_model(model, env, n_eval_episodes=50):
    """Évalue le modèle et retourne des stats utiles."""
    all_rewards = []
    all_lengths = []
    all_step_rewards = []

    for ep in range(n_eval_episodes):
        obs, _ = env.reset()
        done = False
        total_reward = 0
        steps = 0

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, _ = env.step(action)
            total_reward += reward
            steps += 1
            all_step_rewards.append(reward)

        all_rewards.append(total_reward)
        all_lengths.append(steps)

    mean_reward = np.mean(all_rewards)
    std_reward = np.std(all_rewards)
    mean_len = np.mean(all_lengths)
    reward_per_step = np.mean(all_step_rewards)

    print(f"🎯 mean reward = {mean_reward:.4f} ± {std_reward:.4f}")
    print(f"📏 mean episode length = {mean_len:.1f}")
    print(f"⚡ mean reward/step = {reward_per_step:.6f}")
    return mean_reward, std_reward, mean_len, reward_per_step


# Charger ton environnement d’évaluation
eval_env = DummyVecEnv([lambda: ForexEnv()])

# Charger VecNormalize si utilisé pendant l’entraînement
vecnorm_path = "./models/orion_finetune/vecnormalize.pkl"
if os.path.exists(vecnorm_path):
    eval_env = VecNormalize.load(vecnorm_path, eval_env)
    eval_env.training = False
    eval_env.norm_reward = False
    print("✅ VecNormalize chargé")

# Charger les deux modèles
base_model = PPO.load("models/orion/base/ppo_orion", env=eval_env)
tuned_model = PPO.load("models/orion/after_phase_A/ppo_orion", env=eval_env)

print("=== 🔹 Base model performance ===")
base_stats = evaluate_model(base_model, eval_env.envs[0])

print("\n=== 🔸 Fine-tuned model performance ===")
tuned_stats = evaluate_model(tuned_model, eval_env.envs[0])

# Comparaison visuelle
diff = tuned_stats[0] - base_stats[0]
print(f"\n📊 Variation de la reward moyenne = {diff:+.4f}")


## test

In [4]:
"""
rllib_finetune_final.py
Script de fine-tuning multi-phase compatible RLlib 2.x (>=2.30 / 2.50).
Adapté pour éviter les erreurs liées aux changements d'API.
"""

import os
import numpy as np
import ray
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.utils.framework import try_import_torch

torch, _ = try_import_torch()

# -----------------------
# 0) Paramètres
# -----------------------
SAVE_DIR = "./models/rllib_forex"
os.makedirs(SAVE_DIR, exist_ok=True)

# -----------------------
# 1) Env (remplace par ton ForexEnv si tu veux)
# -----------------------
class ForexEnv(gym.Env):
    def __init__(self, config=None):
        super().__init__()
        self.initial_balance = 1000.0
        self.balance = self.initial_balance
        self.prices = [1.1, 1.1, 1.1]
        self.position_size = 1000.0
        self.action_space = gym.spaces.Discrete(3)
        self.observation_space = gym.spaces.Box(low=0.0, high=10.0, shape=(3,), dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        self.balance = self.initial_balance
        self.prices = [1.1, 1.1, 1.1]
        return np.array(self.prices, dtype=np.float32), {}

    def step(self, action):
        old_price = self.prices[-1]
        new_price = float(old_price + np.random.normal(0, 0.01))
        self.prices.append(new_price)
        if len(self.prices) > 3:
            self.prices.pop(0)

        price_change_rel = (new_price - old_price) / old_price
        if action == 0:  # Buy
            reward = price_change_rel * self.position_size
        elif action == 1:  # Sell
            reward = -price_change_rel * self.position_size
        else:
            reward = -1.0

        reward = float(np.clip(reward, -1e3, 1e3))
        self.balance += reward
        done = bool(self.balance <= 0.0 or self.balance >= 2 * self.initial_balance)
        return np.array(self.prices, dtype=np.float32), reward, done, False, {}

# -----------------------
# 2) Init Ray
# -----------------------
ray.shutdown()
ray.init(ignore_reinit_error=True, include_dashboard=False)

# -----------------------
# 3) Config de base (PPOConfig moderne)
# -----------------------
base_config = (
    PPOConfig()
    .environment(env=ForexEnv)          # ton env
    .env_runners(num_env_runners=1)     # remplace rollouts()
    .framework("torch")
    .training(
        gamma=0.99,
        lr=3e-5,
        train_batch_size=4096,
        entropy_coeff=0.005,
        vf_loss_coeff=0.5,
        clip_param=0.15,
    )
    .resources(num_gpus=0)
    .debugging(log_level="INFO")
)

# Build initial algo
algo = base_config.build()
print("✅ PPO initialisé.")

# -----------------------
# 4) Phases de fine-tuning
# -----------------------
phases = [
    {"name": "A_explore", "timesteps": 100_000, "lr": 5e-5, "entropy_coeff": 0.01},
    {"name": "B_stabilize", "timesteps": 200_000, "lr": 3e-5, "entropy_coeff": 0.005},
    {"name": "C_exploit", "timesteps": 200_000, "lr": 1e-5, "entropy_coeff": 0.001},
]

# -----------------------
# 5) Helper: robust parsing of result
# -----------------------
def parse_result_for_steps_and_reward(result, algo_obj):
    """
    Retourne (steps_this_iter, reward_mean). Supporte plusieurs versions RLlib.
    """
    # find steps
    step_keys = [
        "num_env_steps_trained",
        "num_env_steps_sampled",
        "timesteps_this_iter",
        "timesteps_total",
        "num_steps_trained",
    ]
    steps = None
    for k in step_keys:
        if k in result and isinstance(result[k], (int, float)):
            steps = int(result[k])
            break
    # nested fallbacks
    if steps is None:
        for container in ("env_runners", "env_runner", "info", "stats", "custom_metrics"):
            nested = result.get(container)
            if isinstance(nested, dict):
                for k in step_keys:
                    if k in nested and isinstance(nested[k], (int, float)):
                        steps = int(nested[k])
                        break
            if steps is not None:
                break
    # final fallback to algo attributes
    if steps is None:
        steps = getattr(algo_obj, "num_env_steps_sampled", None) or getattr(algo_obj, "num_env_steps_trained", None) or 0
        try:
            steps = int(steps)
        except Exception:
            steps = 0

    # find reward_mean
    reward = None
    for k in ("episode_reward_mean", "reward_mean", "policy_reward_mean"):
        if k in result:
            reward = result[k]
            break
    if reward is None:
        for container in ("env_runners", "env_runner", "info", "stats", "custom_metrics"):
            nested = result.get(container)
            if isinstance(nested, dict) and "episode_reward_mean" in nested:
                reward = nested["episode_reward_mean"]
                break
    return steps, reward

# -----------------------
# 6) Sauvegardes multi-phase corrigées
# -----------------------
import pathlib

_total_global_steps = 0
for phase in phases:
    print(f"\n--- Starting phase {phase['name']} | timesteps={phase['timesteps']:,} lr={phase['lr']} ent={phase['entropy_coeff']}")
    phase_config = base_config.copy(copy_frozen=False).training(lr=phase["lr"], entropy_coeff=phase["entropy_coeff"])

    new_algo = phase_config.build()
    try:
        state = algo.get_state()
        new_algo.set_state(state)
        print("State transferred via get_state()/set_state().")
    except Exception as e:
        try:
            w = algo.get_weights()
            new_algo.set_weights(w)
            print("State transferred via get_weights()/set_weights(). (fallback)")
        except Exception:
            print("⚠️ Warning: could not transfer state automatically:", e)

    total_steps_phase = 0
    first_result_shown = False
    while total_steps_phase < phase["timesteps"]:
        result = new_algo.train()
        steps_this_iter, reward_mean = parse_result_for_steps_and_reward(result, new_algo)
        total_steps_phase += steps_this_iter
        _total_global_steps += steps_this_iter

        if not first_result_shown:
            print("DEBUG: top-level result keys:", list(result.keys()))
            for cont in ("env_runners", "env_runner", "info", "stats", "custom_metrics"):
                if cont in result and isinstance(result[cont], dict):
                    print(f"DEBUG: nested '{cont}' keys:", list(result[cont].keys()))
            first_result_shown = True

        print(f"[{phase['name']}] phase_steps={total_steps_phase:,} (+{steps_this_iter}) reward_mean={reward_mean}")

    # ✅ Chemin absolu sans URI (compatible Windows)
    phase_dir = pathlib.Path(SAVE_DIR) / phase['name']
    phase_dir.mkdir(parents=True, exist_ok=True)
    ckpt = new_algo.save(str(phase_dir.resolve()))
    print(f"Checkpoint saved: {ckpt}")

    algo.stop()
    algo = new_algo

print(f"\nAll phases done. global steps = {_total_global_steps:,}")

# ✅ Sauvegarde finale (corrigée)
final_dir = pathlib.Path(SAVE_DIR) / "final"
final_dir.mkdir(parents=True, exist_ok=True)
final_ckpt = algo.save(str(final_dir.resolve()))
print(f"Final checkpoint: {final_ckpt}")

# -----------------------
# 7) Simple evaluation (local env episodes)
# -----------------------
print("\n=== Quick evaluation (50 episodes) ===")

env = ForexEnv()
rewards = []

# Récupère la policy quelle que soit la version
policy = None

# --- Nouveau stack (Ray ≥ 2.5) ---
if hasattr(algo, "env_runner"):
    runner = algo.env_runner
    if hasattr(runner, "policy"):
        policy = runner.policy
    elif hasattr(runner, "_policy"):
        policy = runner._policy
    elif hasattr(runner, "get_policy"):
        try:
            policy = runner.get_policy()
        except Exception:
            pass

# --- Ancien stack (Ray ≤ 2.4) ---
if policy is None and hasattr(algo, "workers"):
    try:
        # Ancienne API
        policy = algo.get_policy()
    except Exception:
        workers = algo.workers
        # Parfois .local_worker est une propriété, pas une méthode
        worker = workers.local_worker if hasattr(workers, "local_worker") else workers.local_worker()
        policy = worker.get_policy()

if policy is None:
    raise RuntimeError("❌ Impossible de trouver la policy (incompatibilité RLlib version).")

print(f"✅ Policy extraite : {type(policy)}")

for ep in range(50):
    obs, _ = env.reset()
    done = False
    tot = 0.0

    while not done:
        # RLlib attend des numpy arrays, pas forcément des tensors
        obs_arr = np.asarray(obs, dtype=np.float32)
        try:
            action, _, _ = policy.compute_single_action(obs_arr)
        except Exception:
            # Compatibilité pour certaines versions
            out = policy.compute_single_action(torch.as_tensor(obs_arr).unsqueeze(0))
            action = out["actions"].item() if isinstance(out, dict) else out

        # Step (Gym API v26+)
        out = env.step(int(action))
        if len(out) == 5:
            obs, reward, terminated, truncated, info = out
            done = terminated or truncated
        else:
            obs, reward, done, info = out
        tot += float(reward)

    rewards.append(tot)

print(f"Eval mean reward = {np.mean(rewards):.4f} ± {np.std(rewards):.4f}")

algo.stop()
ray.shutdown()


2025-10-19 01:04:13,959	INFO worker.py:2013 -- Started a local Ray instance.
2025-10-19 01:04:32,942	INFO env_runner_group.py:318 -- Inferred observation/action spaces from remote worker (local worker has no env): {'__env__': (Box(0.0, 10.0, (1, 3), float32), MultiDiscrete([3])), '__env_single__': (Box(0.0, 10.0, (3,), float32), Discrete(3)), 'default_policy': (Box(0.0, 10.0, (3,), float32), Discrete(3))}
(SingleAgentEnvRunner pid=47800) DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!
2025-10-19 01:04:33,045	INFO connector_pipeline_v2.py:272 -- Added AddObservationsFromEpisodesToBatch to the end of EnvToModulePipeline.
2025-10-19 01:04:33,093	INFO connector_pipeline_v2.py:272 -- Added AddTimeDimToBatchAndZeroPad to the end of EnvToModulePipeline.
2025-10-19 01:04:33,124	INFO connector_pipeline_v2

✅ PPO initialisé.

--- Starting phase A_explore | timesteps=100,000 lr=5e-05 ent=0.01


2025-10-19 01:04:39,210	INFO env_runner_group.py:318 -- Inferred observation/action spaces from remote worker (local worker has no env): {'__env__': (Box(0.0, 10.0, (1, 3), float32), MultiDiscrete([3])), '__env_single__': (Box(0.0, 10.0, (3,), float32), Discrete(3)), 'default_policy': (Box(0.0, 10.0, (3,), float32), Discrete(3))}
2025-10-19 01:04:39,252	INFO connector_pipeline_v2.py:272 -- Added AddObservationsFromEpisodesToBatch to the end of EnvToModulePipeline.
(SingleAgentEnvRunner pid=43900) DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!
2025-10-19 01:04:39,280	INFO connector_pipeline_v2.py:272 -- Added AddTimeDimToBatchAndZeroPad to the end of EnvToModulePipeline.
2025-10-19 01:04:39,303	INFO connector_pipeline_v2.py:272 -- Added AddStatesFromEpisodesToBatch to the end of EnvToModulePipeli

State transferred via get_state()/set_state().
DEBUG: top-level result keys: ['timers', 'env_runners', 'learners', 'num_training_step_calls_per_iteration', 'num_env_steps_sampled_lifetime', 'fault_tolerance', 'env_runner_group', 'done', 'training_iteration', 'trial_id', 'date', 'timestamp', 'time_this_iter_s', 'time_total_s', 'pid', 'hostname', 'node_ip', 'config', 'time_since_restore', 'iterations_since_restore', 'perf']
DEBUG: nested 'env_runners' keys: ['env_to_module_connector', 'module_to_env_connector', 'num_env_steps_sampled', 'num_agent_steps_sampled_lifetime', 'num_agent_steps_sampled', 'num_env_steps_sampled_lifetime', 'episode_return_max', 'agent_episode_return_mean', 'episode_return_min', 'weights_seq_no', 'env_step_timer', 'episode_len_mean', 'module_episode_return_mean', 'sample', 'num_episodes', 'env_to_module_sum_episodes_length_out', 'env_to_module_sum_episodes_length_in', 'episode_len_min', 'num_module_steps_sampled', 'rlmodule_inference_timer', 'num_episodes_lifetime

2025-10-19 01:10:26,582	INFO env_runner_group.py:318 -- Inferred observation/action spaces from remote worker (local worker has no env): {'__env__': (Box(0.0, 10.0, (1, 3), float32), MultiDiscrete([3])), '__env_single__': (Box(0.0, 10.0, (3,), float32), Discrete(3)), 'default_policy': (Box(0.0, 10.0, (3,), float32), Discrete(3))}
2025-10-19 01:10:26,608	INFO connector_pipeline_v2.py:272 -- Added AddObservationsFromEpisodesToBatch to the end of EnvToModulePipeline.
(SingleAgentEnvRunner pid=38208) DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!
2025-10-19 01:10:26,632	INFO connector_pipeline_v2.py:272 -- Added AddTimeDimToBatchAndZeroPad to the end of EnvToModulePipeline.
2025-10-19 01:10:26,658	INFO connector_pipeline_v2.py:272 -- Added AddStatesFromEpisodesToBatch to the end of EnvToModulePipeli

State transferred via get_state()/set_state().
DEBUG: top-level result keys: ['env_runners', 'learners', 'num_training_step_calls_per_iteration', 'timers', 'num_env_steps_sampled_lifetime', 'fault_tolerance', 'env_runner_group', 'done', 'training_iteration', 'trial_id', 'date', 'timestamp', 'time_this_iter_s', 'time_total_s', 'pid', 'hostname', 'node_ip', 'config', 'time_since_restore', 'iterations_since_restore', 'perf']
DEBUG: nested 'env_runners' keys: ['agent_episode_return_mean', 'env_reset_timer', 'env_step_timer', 'env_to_module_connector', 'env_to_module_sum_episodes_length_in', 'env_to_module_sum_episodes_length_out', 'episode_duration_sec_mean', 'episode_len_max', 'episode_len_mean', 'episode_len_min', 'episode_return_max', 'episode_return_mean', 'episode_return_min', 'module_episode_return_mean', 'module_to_env_connector', 'num_agent_steps_sampled', 'num_agent_steps_sampled_lifetime', 'num_env_steps_sampled', 'num_env_steps_sampled_lifetime', 'num_episodes', 'num_episodes_li

2025-10-19 01:21:18,320	INFO env_runner_group.py:318 -- Inferred observation/action spaces from remote worker (local worker has no env): {'__env__': (Box(0.0, 10.0, (1, 3), float32), MultiDiscrete([3])), '__env_single__': (Box(0.0, 10.0, (3,), float32), Discrete(3)), 'default_policy': (Box(0.0, 10.0, (3,), float32), Discrete(3))}
(SingleAgentEnvRunner pid=41476) DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!
2025-10-19 01:21:18,351	INFO connector_pipeline_v2.py:272 -- Added AddObservationsFromEpisodesToBatch to the end of EnvToModulePipeline.
2025-10-19 01:21:18,378	INFO connector_pipeline_v2.py:272 -- Added AddTimeDimToBatchAndZeroPad to the end of EnvToModulePipeline.
2025-10-19 01:21:18,404	INFO connector_pipeline_v2.py:272 -- Added AddStatesFromEpisodesToBatch to the end of EnvToModulePipeli

State transferred via get_state()/set_state().
DEBUG: top-level result keys: ['env_runners', 'learners', 'num_training_step_calls_per_iteration', 'timers', 'num_env_steps_sampled_lifetime', 'fault_tolerance', 'env_runner_group', 'done', 'training_iteration', 'trial_id', 'date', 'timestamp', 'time_this_iter_s', 'time_total_s', 'pid', 'hostname', 'node_ip', 'config', 'time_since_restore', 'iterations_since_restore', 'perf']
DEBUG: nested 'env_runners' keys: ['agent_episode_return_mean', 'env_reset_timer', 'env_step_timer', 'env_to_module_connector', 'env_to_module_sum_episodes_length_in', 'env_to_module_sum_episodes_length_out', 'episode_duration_sec_mean', 'episode_len_max', 'episode_len_mean', 'episode_len_min', 'episode_return_max', 'episode_return_mean', 'episode_return_min', 'module_episode_return_mean', 'module_to_env_connector', 'num_agent_steps_sampled', 'num_agent_steps_sampled_lifetime', 'num_env_steps_sampled', 'num_env_steps_sampled_lifetime', 'num_episodes', 'num_episodes_li

AttributeError: 'function' object has no attribute 'local_worker'